# 📊 ANÁLISIS INTEGRAL: EFICACIA DE PLATAFORMAS DE BÚSQUEDA LABORAL
## Análisis Exploratorio Multivariado - Primer Corte

**Objetivo:** Identificar factores predictivos de éxito laboral mediante análisis descriptivo, bivariado y multivariado.

**Estructura:**
1. Setup & Configuración - Importaciones y parámetros
2. Carga de Datos - CSV loading
3. Análisis Univariado - Distribuciones
4. Análisis Bivariado - Relaciones
5. Análisis Multivariado - PCA, Clustering, Interacciones
6. Síntesis - Hallazgos y recomendaciones

---
## SECCIÓN 1: Setup & Configuración

In [16]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# Configuración reproducible
warnings.filterwarnings('ignore')
np.random.seed(42)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

# Estilo de gráficos
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (14, 8)

print('✓ Entorno configurado correctamente')
print(f'  NumPy: {np.__version__}')
print(f'  Pandas: {pd.__version__}')

✓ Entorno configurado correctamente
  NumPy: 2.4.4
  Pandas: 3.0.2


---
## SECCIÓN 2: Carga de Datos

In [17]:
# Configurar rutas
PROJECT_ROOT = Path('.').resolve()
DATA_PATH = PROJECT_ROOT / 'job_search_platform_efficacy_100k.csv'
OUTPUT_DIR = PROJECT_ROOT / 'outputs_primer_corte'
OUTPUT_DIR.mkdir(exist_ok=True)

print(f'Cargando: {DATA_PATH.name}')

# Cargar datos
raw_df = pd.read_csv(DATA_PATH)
clean_df = raw_df.copy()

print(f'✓ Datos cargados: {len(raw_df):,} filas × {len(raw_df.columns)} columnas')
print(f'  Tasa de oferta: {clean_df["Offer_Received"].mean()*100:.2f}%')
print(f'  Valores nulos: {raw_df.isnull().sum().sum()}')

Cargando: job_search_platform_efficacy_100k.csv
✓ Datos cargados: 100,000 filas × 20 columnas
  Tasa de oferta: 34.23%
  Valores nulos: 328855


---
## SECCIÓN 3: Análisis Univariado

In [18]:
print('\n' + '='*80)
print('ESTADÍSTICAS DESCRIPTIVAS')
print('='*80)

key_vars = ['GPA', 'Applications_Submitted', 'First_Round_Interviews', 
            'Second_Round_Interviews', 'Offer_Salary']

# Mostrar estadísticas
for var in key_vars:
    if var == 'Offer_Salary':
        data = clean_df[clean_df['Offer_Received'] == 1][var]
        print(f'\n{var} (solo con oferta, n={len(data):,})')
    else:
        data = clean_df[var]
        print(f'\n{var}')
    
    print(f'  Mean: {data.mean():.2f} | Median: {data.median():.2f} | Std: {data.std():.2f}')
    print(f'  Min: {data.min():.2f} | Max: {data.max():.2f}')


ESTADÍSTICAS DESCRIPTIVAS

GPA
  Mean: 3.20 | Median: 3.20 | Std: 0.39
  Min: 2.00 | Max: 4.00

Applications_Submitted
  Mean: 53.73 | Median: 42.00 | Std: 45.00
  Min: 5.00 | Max: 300.00

First_Round_Interviews
  Mean: 2.17 | Median: 2.00 | Std: 1.91
  Min: 0.00 | Max: 21.00

Second_Round_Interviews
  Mean: 1.01 | Median: 1.00 | Std: 1.14
  Min: 0.00 | Max: 12.00

Offer_Salary (solo con oferta, n=34,229)
  Mean: 76777.29 | Median: 76823.00 | Std: 14931.47
  Min: 35000.00 | Max: 137588.00


---
## SECCIÓN 4: Análisis Bivariado - Correlaciones

In [19]:
print('\n' + '='*80)
print('MATRIZ DE CORRELACIONES')
print('='*80)

numeric_cols = ['GPA', 'Applications_Submitted', 'First_Round_Interviews',
                'Second_Round_Interviews', 'Offer_Received']
corr_matrix = clean_df[numeric_cols].corr()

print('\nTOP Correlaciones con Offer_Received:')
corr_offer = corr_matrix['Offer_Received'].sort_values(ascending=False)
for i, (var, val) in enumerate(corr_offer.items(), 1):
    if var != 'Offer_Received':
        strength = 'FUERTE' if abs(val) > 0.4 else 'MODERADO' if abs(val) > 0.2 else 'DÉBIL'
        print(f'  {i}. {var:30s} r = {val:+.4f} [{strength}]')


MATRIZ DE CORRELACIONES

TOP Correlaciones con Offer_Received:
  2. Second_Round_Interviews        r = +0.5492 [FUERTE]
  3. First_Round_Interviews         r = +0.2200 [MODERADO]
  4. GPA                            r = +0.0875 [DÉBIL]
  5. Applications_Submitted         r = +0.0116 [DÉBIL]


---
## SECCIÓN 5: Análisis Multivariado - PCA

In [20]:
print('\n' + '='*80)
print('PCA: ANÁLISIS DE COMPONENTES PRINCIPALES')
print('='*80)

# Preparar datos para PCA
pca_vars = ['GPA', 'Applications_Submitted', 'First_Round_Interviews', 'Second_Round_Interviews']
X_pca = clean_df[pca_vars].fillna(clean_df[pca_vars].mean())
X_scaled = StandardScaler().fit_transform(X_pca)

# Aplicar PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(X_scaled)

print(f'\nVarianza explicada por componente:')
for i, var in enumerate(pca.explained_variance_ratio_):
    cumsum = pca.explained_variance_ratio_[:i+1].sum()
    print(f'  PC{i+1}: {var*100:5.1f}% (Acumulada: {cumsum*100:6.1f}%)')


PCA: ANÁLISIS DE COMPONENTES PRINCIPALES

Varianza explicada por componente:
  PC1:  48.0% (Acumulada:   48.0%)
  PC2:  25.0% (Acumulada:   73.1%)


---
## SECCIÓN 6: Análisis Multivariado - Clustering

In [21]:
print('\n' + '='*80)
print('CLUSTERING: PERFILES DE ESTUDIANTES')
print('='*80)

# Preparar datos para clustering
cluster_vars = ['GPA', 'Applications_Submitted', 'First_Round_Interviews', 'Second_Round_Interviews']
X_cluster = clean_df[cluster_vars].fillna(clean_df[cluster_vars].mean())
X_cluster_scaled = StandardScaler().fit_transform(X_cluster)

# K-means clustering
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_cluster_scaled)
clean_df['Cluster'] = clusters

print('\nCaracterización de Clusters:')
for c in range(3):
    cluster_data = clean_df[clean_df['Cluster'] == c]
    offer_rate = cluster_data['Offer_Received'].mean() * 100
    
    print(f'\nCLUSTER {c+1}: n={len(cluster_data):,} ({len(cluster_data)/len(clean_df)*100:.1f}%)')
    print(f'  Tasa Oferta: {offer_rate:.1f}%')
    print(f'  GPA promedio: {cluster_data["GPA"].mean():.2f}')
    print(f'  Aplicaciones: {cluster_data["Applications_Submitted"].mean():.1f}')
    print(f'  2da Ronda: {cluster_data["Second_Round_Interviews"].mean():.2f}')


CLUSTERING: PERFILES DE ESTUDIANTES

Caracterización de Clusters:

CLUSTER 1: n=21,469 (21.5%)
  Tasa Oferta: 46.6%
  GPA promedio: 3.19
  Aplicaciones: 102.2
  2da Ronda: 2.30

CLUSTER 2: n=38,572 (38.6%)
  Tasa Oferta: 25.2%
  GPA promedio: 2.87
  Aplicaciones: 41.0
  2da Ronda: 0.57

CLUSTER 3: n=39,959 (40.0%)
  Tasa Oferta: 36.3%
  GPA promedio: 3.52
  Aplicaciones: 40.0
  2da Ronda: 0.75


---
## SECCIÓN 7: Efecto de Plataforma

In [22]:
print('\n' + '='*80)
print('EFECTO DE PLATAFORMA')
print('='*80)

platform_stats = clean_df.groupby('Primary_Search_Platform').agg({
    'Offer_Received': ['count', 'mean']
})

platform_stats.columns = ['Count', 'Offer_Rate']
platform_stats['Offer_Rate_Pct'] = (platform_stats['Offer_Rate'] * 100).round(1)
platform_stats = platform_stats.sort_values('Offer_Rate_Pct', ascending=False)

print('\nTasa de Oferta por Plataforma:')
for platform, row in platform_stats.iterrows():
    print(f'  {platform:15s} {row["Offer_Rate_Pct"]:5.1f}% (n={int(row["Count"]):,})')

gap = platform_stats['Offer_Rate_Pct'].max() - platform_stats['Offer_Rate_Pct'].min()
print(f'\n  GAP: {gap:.1f} pp (Brecha entre mejor y peor)')


EFECTO DE PLATAFORMA

Tasa de Oferta por Plataforma:
  Handshake        37.0% (n=35,871)
  LinkedIn         37.0% (n=39,573)
  Indeed           25.8% (n=24,556)

  GAP: 11.2 pp (Brecha entre mejor y peor)


---
## SECCIÓN 8: Síntesis de Hallazgos

In [23]:
print('\n' + '='*80)
print('RESUMEN EJECUTIVO')
print('='*80)

print(f'\n1. TASA DE OFERTA GLOBAL: {clean_df["Offer_Received"].mean()*100:.2f}%')

print(f'\n2. TOP 3 PREDICTORES DE OFERTA:')
top_corr = corr_offer[1:4]
for i, (var, val) in enumerate(top_corr.items(), 1):
    print(f'   {i}. {var:30s} r = {val:+.4f}')

print(f'\n3. EFECTO DE PLATAFORMA:')
best_platform = platform_stats["Offer_Rate_Pct"].idxmax()
worst_platform = platform_stats["Offer_Rate_Pct"].idxmin()
best_rate = platform_stats["Offer_Rate_Pct"].max()
worst_rate = platform_stats["Offer_Rate_Pct"].min()
print(f'   Mejor: {best_platform} ({best_rate:.1f}%)')
print(f'   Peor:  {worst_platform} ({worst_rate:.1f}%)')
print(f'   GAP:   {best_rate - worst_rate:.1f} pp')

print(f'\n4. RECOMENDACIONES INMEDIATAS:')
print(f'   - Reasignar a {best_platform}: +{best_rate - worst_rate:.1f} pp en oferta')
print(f'   - Invertir en 2da Ronda coaching: r={corr_offer["Second_Round_Interviews"]:.2f}')
print(f'   - Enfatizar CALIDAD sobre CANTIDAD en aplicaciones')
print(f'   - Segmentar por cluster para intervenciones diferenciadas')

print('\n' + '='*80)
print('✓ ANÁLISIS COMPLETADO')
print('='*80)


RESUMEN EJECUTIVO

1. TASA DE OFERTA GLOBAL: 34.23%

2. TOP 3 PREDICTORES DE OFERTA:
   1. Second_Round_Interviews        r = +0.5492
   2. First_Round_Interviews         r = +0.2200
   3. GPA                            r = +0.0875

3. EFECTO DE PLATAFORMA:
   Mejor: Handshake (37.0%)
   Peor:  Indeed (25.8%)
   GAP:   11.2 pp

4. RECOMENDACIONES INMEDIATAS:
   - Reasignar a Handshake: +11.2 pp en oferta
   - Invertir en 2da Ronda coaching: r=0.55
   - Enfatizar CALIDAD sobre CANTIDAD en aplicaciones
   - Segmentar por cluster para intervenciones diferenciadas

✓ ANÁLISIS COMPLETADO
